<a href="https://colab.research.google.com/github/fergogu27-ctrl/EDPII/blob/main/Tarea_1_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# MODELO DE OPTIMIZACIÓN DE PRODUCCIÓN: TECH-COMPONENTS S.A.
# Programación lineal con 10 variables

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import linprog

# 1. Definicion de la Funcion objetivo
# Max Z = 40x1 + 65x2 + 35x3 + 90x4
#         - 15x5 - 20x6
#         - 10x7 - 12x8 - 8x9 - 15x10
#
# usamos c = -coeficientes de la función objetivo

c_max = np.array([40, 65, 35, 90, -15, -20, -10, -12, -8, -15], dtype=float)
c = -c_max



# 2. Restricciones A_ub x <= b_ub
# Variables:
# x1, x2, x3, x4 = unidades de sensores A, B, C, D
# x5 = horas extra en ensamble
# x6 = horas extra en pruebas
# x7, x8, x9, x10 = materia prima externa para A, B, C, D

A_ub = np.array([
    # 2x1 + 3x2 + 1.5x3 + 4x4 - x5 <= 800
    [2, 3, 1.5, 4, -1, 0, 0, 0, 0, 0],

    # x1 + 2x2 + 2x3 + x4 - x6 <= 500
    [1, 2, 2, 1, 0, -1, 0, 0, 0, 0],

    # x5 <= 100
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],

    # x6 <= 100
    [0, 0, 0, 0, 0, 1, 0, 0, 0, 0],

    # x1 >= 50  ->  -x1 <= -50
    [-1, 0, 0, 0, 0, 0, 0, 0, 0, 0],

    # x2 >= 40  ->  -x2 <= -40
    [0, -1, 0, 0, 0, 0, 0, 0, 0, 0],

    # x4 >= 30  ->  -x4 <= -30
    [0, 0, 0, -1, 0, 0, 0, 0, 0, 0],

    # 0.5x1 + 0.8x2 + 0.4x3 + 1.2x4 - x7 - x8 - x9 - x10 <= 300
    [0.5, 0.8, 0.4, 1.2, 0, 0, -1, -1, -1, -1],

    # x7 <= 0.5x1  ->  x7 - 0.5x1 <= 0
    [-0.5, 0, 0, 0, 0, 0, 1, 0, 0, 0],

    # x8 <= 0.8x2  ->  x8 - 0.8x2 <= 0
    [0, -0.8, 0, 0, 0, 0, 0, 1, 0, 0],

    # x9 <= 0.4x3  ->  x9 - 0.4x3 <= 0
    [0, 0, -0.4, 0, 0, 0, 0, 0, 1, 0],

    # x10 <= 1.2x4 ->  x10 - 1.2x4 <= 0
    [0, 0, 0, -1.2, 0, 0, 0, 0, 0, 1],

    # 15x5 + 20x6 + 10x7 + 12x8 + 8x9 + 15x10 <= 5000
    [0, 0, 0, 0, 15, 20, 10, 12, 8, 15]
], dtype=float)

b_ub = np.array([
    800,
    500,
    100,
    100,
    -50,
    -40,
    -30,
    300,
    0,
    0,
    0,
    0,
    5000
], dtype=float)



# 3. Cotas de las variables
# No negatividad: xi >= 0
bounds = [(0, None)] * 10



# 4. Solución
resultado = linprog(
    c=c,
    A_ub=A_ub,
    b_ub=b_ub,
    bounds=bounds,
    method="highs"
)



# 5. MOSTRAR LA SOLUCIÓN

if resultado.success:
    x = resultado.x
    z_opt = c_max @ x

    print("Se encontró solución óptima.\n")

    nombres = [
        "x1 = Sensores A",
        "x2 = Sensores B",
        "x3 = Sensores C",
        "x4 = Sensores D",
        "x5 = Hrs extra ensamble",
        "x6 = Hrs extra pruebas",
        "x7 = Materia externa A",
        "x8 = Materia externa B",
        "x9 = Materia externa C",
        "x10 = Materia externa D"
    ]

    for nombre, valor in zip(nombres, x):
        print(f"{nombre:25s}: {valor:.4f}")

    print(f"\nValor óptimo de Z = {z_opt:.4f}")

else:
    print("No se encontró solución óptima.")
    print(resultado.message)



# 6. Resultados en un tabla para mayor visualizacion

if resultado.success:
    tabla = pd.DataFrame({
        "Variable": [f"x{i}" for i in range(1, 11)],
        "Interpretación": [
            "Producción A",
            "Producción B",
            "Producción C",
            "Producción D",
            "Horas extra ensamble",
            "Horas extra pruebas",
            "Materia externa A",
            "Materia externa B",
            "Materia externa C",
            "Materia externa D",
        ],
        "Valor óptimo": x
    })

    print("\nTabla de solución:")
    print(tabla.to_string(index=False))



# 7. Comprobación de restricciones

if resultado.success:
    x1,x2,x3,x4,x5,x6,x7,x8,x9,x10 = x

    verificacion = [
        ["Ensamblaje", 2*x1 + 3*x2 + 1.5*x3 + 4*x4, "<=", 800 + x5],
        ["Pruebas", x1 + 2*x2 + 2*x3 + x4, "<=", 500 + x6],
        ["Tiempo extra ensamble", x5, "<=", 100],
        ["Tiempo extra pruebas", x6, "<=", 100],
        ["Demanda mínima A", x1, ">=", 50],
        ["Demanda mínima B", x2, ">=", 40],
        ["Demanda mínima D", x4, ">=", 30],
        ["Materia prima total", 0.5*x1 + 0.8*x2 + 0.4*x3 + 1.2*x4, "<=", 300 + x7 + x8 + x9 + x10],
        ["Materia externa A", x7, "<=", 0.5*x1],
        ["Materia externa B", x8, "<=", 0.8*x2],
        ["Materia externa C", x9, "<=", 0.4*x3],
        ["Materia externa D", x10, "<=", 1.2*x4],
        ["Presupuesto extra", 15*x5 + 20*x6 + 10*x7 + 12*x8 + 8*x9 + 15*x10, "<=", 5000]
    ]

    tabla_ver = pd.DataFrame(
        verificacion,
        columns=["Restricción", "Lado izquierdo", "Signo", "Lado derecho"]
    )

    print("\nVerificación de restricciones:")
    print(tabla_ver.to_string(index=False))




Se encontró solución óptima.

x1 = Sensores A          : 50.0000
x2 = Sensores B          : 40.0000
x3 = Sensores C          : 123.0769
x4 = Sensores D          : 123.8462
x5 = Hrs extra ensamble  : 100.0000
x6 = Hrs extra pruebas   : 0.0000
x7 = Materia externa A   : 0.0000
x8 = Materia externa B   : 0.0000
x9 = Materia externa C   : 0.0000
x10 = Materia externa D  : 0.0000

Valor óptimo de Z = 18553.8462

Tabla de solución:
Variable       Interpretación  Valor óptimo
      x1         Producción A     50.000000
      x2         Producción B     40.000000
      x3         Producción C    123.076923
      x4         Producción D    123.846154
      x5 Horas extra ensamble    100.000000
      x6  Horas extra pruebas      0.000000
      x7    Materia externa A      0.000000
      x8    Materia externa B      0.000000
      x9    Materia externa C      0.000000
     x10    Materia externa D      0.000000

Verificación de restricciones:
          Restricción  Lado izquierdo Signo  Lado dere